### Import necessary libraries and install required packages

In [ ]:
!pip install transformers datasets scikit-learn torch pandas pyarrow

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_scheduler
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm.auto import tqdm

# --------- data & tokenizer ----------
train_df = pd.read_csv("train.csv", sep=",")
test_df  = pd.read_csv("test.csv", sep=",")

label_names = ["Joy","Trust","Fear","Surprise","Sadness","Disgust","Anger","Anticipation","Neutral"]
checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --------- custom Dataset (returns raw lists so collator can pad) ----------
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        labels = [int(row[l]) for l in self.label_names]
        tokenized["labels"] = labels
        return tokenized

train_dataset = EmotionDataset(train_df, tokenizer, label_names)
eval_dataset  = EmotionDataset(test_df, tokenizer, label_names)

# --------- collate function that uses HuggingFace padding and stacks labels ----------
def collate_fn(batch):
    # batch: list of dicts from __getitem__
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)                        # returns tensors for input_ids/attention_mask etc
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.float)    # shape: (batch, num_labels)
    return batch_inputs

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
eval_loader  = DataLoader(eval_dataset,  batch_size=16, shuffle=False, collate_fn=collate_fn)

# --------- model, optimizer, scheduler ----------
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_names),
    problem_type="multi_label_classification"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# --------- training loop ----------
progress_bar = tqdm(range(num_training_steps))
model.train()
for epoch in range(num_epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

# --------- evaluation ----------
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in eval_loader:
        labels = batch["labels"].cpu().numpy().astype(int)   # (batch, num_labels)
        # move only the inputs to device
        model_inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        outputs = model(**model_inputs)
        logits = outputs.logits.cpu()
        preds = (torch.sigmoid(logits).numpy() > 0.5).astype(int)   # threshold 0.5
        all_preds.append(preds)
        all_labels.append(labels)

y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

# metrics
print("Subset Accuracy (exact match):", accuracy_score(y_true, y_pred))
print("Micro Precision:", precision_score(y_true, y_pred, average="micro", zero_division=0))
print("Micro Recall:",    recall_score(y_true, y_pred, average="micro", zero_division=0))
print("Micro F1:",        f1_score(y_true, y_pred, average="micro", zero_division=0))
print("Macro Precision:", precision_score(y_true, y_pred, average="macro", zero_division=0))
print("Macro Recall:",    recall_score(y_true, y_pred, average="macro", zero_division=0))
print("Macro F1:",        f1_score(y_true, y_pred, average="macro", zero_division=0))
print("\nPer-class report:\n", classification_report(y_true, y_pred, target_names=label_names, zero_division=0))


## First training results:

- Subset Accuracy (exact match): 0.3607305936073059
- Micro Precision: 0.7622950819672131
- Micro Recall: 0.372
- Micro F1: 0.5
- Macro Precision: 0.27744046414356066
- Macro Recall: 0.16385374445483736
- Macro F1: 0.18947683780848593

Per-class report:
| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.74      | 0.61   | 0.67     | 74      |
| Trust         | 0.00      | 0.00   | 0.00     | 30      |
| Fear          | 0.00      | 0.00   | 0.00     | 5       |
| Surprise      | 0.00      | 0.00   | 0.00     | 11      |
| Sadness       | 0.76      | 0.67   | 0.71     | 61      |
| Disgust       | 0.00      | 0.00   | 0.00     | 10      |
| Anger         | 0.00      | 0.00   | 0.00     | 5       |
| Anticipation  | 1.00      | 0.19   | 0.33     | 36      |
| Neutral       | 0.00      | 0.00   | 0.00     | 18      |

|          | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| **micro avg** | 0.76      | 0.37   | 0.50     | 250     |
| **macro avg** | 0.28      | 0.16   | 0.19     | 250     |
| **weighted avg** | 0.55   | 0.37   | 0.42     | 250     |
| **samples avg**  | 0.42   | 0.39   | 0.40     | 250     |

## Second training results:
- Subset Accuracy (exact match): 0.4063926940639269
- Micro Precision: 0.8015267175572519
- Micro Recall: 0.42
- Micro F1: 0.5511811023622047
- Macro Precision: 0.27762984378170014
- Macro Recall: 0.1942311163622639
- Macro F1: 0.2237746200594498

Per-class report:
| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.81      | 0.68   | 0.74     | 74      |
| Trust         | 0.00      | 0.00   | 0.00     | 30      |
| Fear          | 0.00      | 0.00   | 0.00     | 5       |
| Surprise      | 0.00      | 0.00   | 0.00     | 11      |
| Sadness       | 0.75      | 0.66   | 0.70     | 61      |
| Disgust       | 0.00      | 0.00   | 0.00     | 10      |
| Anger         | 0.00      | 0.00   | 0.00     | 5       |
| Anticipation  | 0.94      | 0.42   | 0.58     | 36      |
| Neutral       | 0.00      | 0.00   | 0.00     | 18      |

|                  | Precision | Recall | F1-score | Support |
|------------------|-----------|--------|----------|---------|
| **micro avg**    | 0.80      | 0.42   | 0.55     | 250     |
| **macro avg**    | 0.28      | 0.19   | 0.22     | 250     |
| **weighted avg** | 0.56   | 0.42   | 0.47     | 250     |
| **samples avg**  | 0.47   | 0.44   | 0.45     | 250     |






# 90-10 train-test split results:

## First run:
- Subset Accuracy (exact match): 0.38181818181818183
- Micro Precision: 0.796875
- Micro Recall: 0.4112903225806452
- Micro F1: 0.5425531914893617
- Macro Precision: 0.28754208754208754
- Macro Recall: 0.17577577577577577
- Macro F1: 0.18520163731606518

Per-class report:
| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.80      | 0.65   | 0.72     | 37      |
| Trust         | 1.00      | 0.07   | 0.12     | 15      |
| Fear          | 0.00      | 0.00   | 0.00     | 2       |
| Surprise      | 0.00      | 0.00   | 0.00     | 5       |
| Sadness       | 0.79      | 0.87   | 0.83     | 30      |
| Disgust       | 0.00      | 0.00   | 0.00     | 5       |
| Anger         | 0.00      | 0.00   | 0.00     | 3       |
| Anticipation  | 0.00      | 0.00   | 0.00     | 18      |
| Neutral       | 0.00      | 0.00   | 0.00     | 9       |

|                  | Precision | Recall | F1-score | Support |
|------------------|-----------|--------|----------|---------|
| **micro avg**    | 0.80      | 0.41   | 0.54     | 124     |
| **macro avg**    | 0.29      | 0.18   | 0.19     | 124     |
| **weighted avg** | 0.55   | 0.41   | 0.43     | 124     |
| **samples avg**  | 0.46   | 0.42   | 0.44     | 124     |

## Second run:

- Subset Accuracy (exact match): 0.4727272727272727
- Micro Precision: 0.8591549295774648
- Micro Recall: 0.49193548387096775
- Micro F1: 0.6256410256410256
- Macro Precision: 0.4040270429159318
- Macro Recall: 0.25795795795795795
- Macro F1: 0.309183603920446

Per-class report:
| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.82      | 0.62   | 0.71     | 37      |
| Trust         | 1.00      | 0.47   | 0.64     | 15      |
| Fear          | 0.00      | 0.00   | 0.00     | 2       |
| Surprise      | 0.00      | 0.00   | 0.00     | 5       |
| Sadness       | 0.81      | 0.73   | 0.77     | 30      |
| Disgust       | 0.00      | 0.00   | 0.00     | 5       |
| Anger         | 0.00      | 0.00   | 0.00     | 3       |
| Anticipation  | 1.00      | 0.50   | 0.67     | 18      |
| Neutral       | 0.00      | 0.00   | 0.00     | 9       |

|                  | Precision | Recall | F1-score | Support |
|------------------|-----------|--------|----------|---------|
| **micro avg**    | 0.86      | 0.49   | 0.63     | 124     |
| **macro avg**    | 0.40      | 0.26   | 0.31     | 124     |
| **weighted avg** | 0.71   | 0.49   | 0.57     | 124     |
| **samples avg**  | 0.55   | 0.51   | 0.53     | 124     |



## Cross validation with K = 10

In [3]:
import sys
!{sys.executable} -m pip install iterative-stratification


[notice] A new release of pip available: 22.3.1 -> 25.2
[notice] To update, run: /Users/carlotacatot/.pyenv/versions/3.11.1/bin/python -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm.auto import tqdm

# Loading the data file
df = pd.read_csv("../GroundTruth.csv", sep=";")
label_names = ["Joy","Trust","Fear","Surprise","Sadness","Disgust","Anger","Anticipation","Neutral"]

# We discard all the reject entries
df = df[df['Reject'] != 1]

# Tokenize & collate
checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create a custom Dataset
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        labels = [int(row[l]) for l in self.label_names]
        tokenized["labels"] = labels
        return tokenized

# Create a custom collate function
def collate_fn(batch):
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)                        # returns tensors for input_ids/attention_mask etc
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.float)    # shape: (batch, num_labels)
    return batch_inputs


# Cross-validation
X = df["sentence"].values
y = df[label_names].values

mskf = MultilabelStratifiedKFold(n_splits=10, shuffle=True, random_state=42)
all_metrics = []

for fold, (train_idx, val_idx) in enumerate(mskf.split(X, y)):
    print(f"\n===== Fold {fold+1} =====")
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = EmotionDataset(train_df, tokenizer, label_names)
    val_dataset   = EmotionDataset(val_df, tokenizer, label_names)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, collate_fn=collate_fn)

    # ---- re-init model each fold ----
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(label_names),
        problem_type="multi_label_classification"
    )
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)
    num_epochs = 3
    num_training_steps = num_epochs * len(train_loader)
    lr_scheduler = get_scheduler("linear", optimizer=optimizer,
                                 num_warmup_steps=0,
                                 num_training_steps=num_training_steps)

    # ---- training loop ----
    model.train()
    progress_bar = tqdm(range(num_training_steps))
    for epoch in range(num_epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

    # ---- evaluation ----
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch["labels"].cpu().numpy().astype(int)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            outputs = model(**inputs)
            logits = outputs.logits.cpu()
            preds = (torch.sigmoid(logits).numpy() > 0.5).astype(int)
            all_preds.append(preds)
            all_labels.append(labels)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    metrics = {
        "subset_acc": accuracy_score(y_true, y_pred),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }
    print(metrics)
    all_metrics.append(metrics)

# ===== Average results =====
df_metrics = pd.DataFrame(all_metrics)
print("\n===== Cross-validation results =====")
print(df_metrics.mean())



===== Fold 1 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 369/369 [04:26<00:00,  1.66it/s]

{'subset_acc': 0.27927927927927926, 'micro_precision': 0.7843137254901961, 'micro_recall': 0.32, 'micro_f1': 0.45454545454545453, 'macro_f1': 0.12203829925348912}

===== Fold 2 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 369/369 [04:28<00:00,  1.37it/s]

100%|██████████| 369/369 [05:55<00:00,  1.13s/it]

{'subset_acc': 0.4351851851851852, 'micro_precision': 0.7236842105263158, 'micro_recall': 0.4365079365079365, 'micro_f1': 0.5445544554455446, 'macro_f1': 0.23212315111049292}

===== Fold 3 =====


...

{'subset_acc': 0.40869565217391307, 'micro_precision': 0.828125, 'micro_recall': 0.4274193548387097, 'micro_f1': 0.5638297872340425, 'macro_f1': 0.2001547987616099}

===== Fold 9 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 369/369 [04:09<00:00,  2.05it/s]

{'subset_acc': 0.35514018691588783, 'micro_precision': 0.6521739130434783, 'micro_recall': 0.3629032258064516, 'micro_f1': 0.46632124352331605, 'macro_f1': 0.17120657448361748}

===== Fold 10 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 369/369 [04:11<00:00,  1.47it/s]

100%|██████████| 369/369 [04:06<00:00,  1.49it/s]

{'subset_acc': 0.34579439252336447, 'micro_precision': 0.7413793103448276, 'micro_recall': 0.3412698412698413, 'micro_f1': 0.4673913043478261, 'macro_f1': 0.14487394957983196}

===== Cross-validation results =====
subset_acc         0.363326
micro_precision    0.743149
micro_recall       0.380889
micro_f1           0.502449
macro_f1           0.175612
dtype: float64


## Cross validation first run solution

- **Fold 1**: {'subset_acc': 0.27927927927927926, 'micro_precision': 0.7843137254901961, 'micro_recall': 0.32, 'micro_f1': 0.45454545454545453, 'macro_f1': 0.12203829925348912}
- **Fold 2**: {'subset_acc': 0.4351851851851852, 'micro_precision': 0.7236842105263158, 'micro_recall': 0.4365079365079365, 'micro_f1': 0.5445544554455446, 'macro_f1': 0.23212315111049292}
- **Fold 3**: {'subset_acc': 0.39814814814814814, 'micro_precision': 0.8412698412698413, 'micro_recall': 0.4274193548387097, 'micro_f1': 0.5668449197860963, 'macro_f1': 0.21672532260767552}
- **Fold 4**: {'subset_acc': 0.3577981651376147, 'micro_precision': 0.676056338028169, 'micro_recall': 0.384, 'micro_f1': 0.4897959183673469, 'macro_f1': 0.18172581516843814}
- **Fold 5**: {'subset_acc': 0.34234234234234234, 'micro_precision': 0.676923076923077, 'micro_recall': 0.352, 'micro_f1': 0.4631578947368421, 'macro_f1': 0.13333333333333333}
- **Fold 6**: {'subset_acc': 0.3394495412844037, 'micro_precision': 0.765625, 'micro_recall': 0.3951612903225806, 'micro_f1': 0.5212765957446809, 'macro_f1': 0.1798218029350105}
- **Fold 7**: {'subset_acc': 0.37142857142857144, 'micro_precision': 0.7419354838709677, 'micro_recall': 0.36220472440944884, 'micro_f1': 0.48677248677248675, 'macro_f1': 0.1741138560687433}
- **Fold 8**: {'subset_acc': 0.40869565217391307, 'micro_precision': 0.828125, 'micro_recall': 0.4274193548387097, 'micro_f1': 0.5638297872340425, 'macro_f1': 0.2001547987616099}
- **Fold 9**: {'subset_acc': 0.35514018691588783, 'micro_precision': 0.6521739130434783, 'micro_recall': 0.3629032258064516, 'micro_f1': 0.46632124352331605, 'macro_f1': 0.17120657448361748}
- **Fold 10**: {'subset_acc': 0.34579439252336447, 'micro_precision': 0.7413793103448276, 'micro_recall': 0.3412698412698413, 'micro_f1': 0.4673913043478261, 'macro_f1': 0.14487394957983196}


===== Cross-validation results =====
- subset_acc         0.363326
- micro_precision    0.743149
- micro_recall       0.380889
- micro_f1           0.502449
- macro_f1           0.175612



In [9]:
import pandas as pd
import numpy as np
import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm.auto import tqdm

# Loading the data file
df = pd.read_csv("../GroundTruth.csv", sep=";")
label_names = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral"]

# We discard all the reject entries
df = df[df['Reject'] != 1]

# Tokenize & collate
checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Create a custom Dataset
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        labels = [int(row[l]) for l in self.label_names]
        tokenized["labels"] = labels
        return tokenized


# Create a custom collate function
def collate_fn(batch):
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.float)
    return batch_inputs


# Cross-validation
X = df["sentence"].values
y = df[label_names].values

mskf = MultilabelStratifiedKFold(n_splits=10, shuffle=True, random_state=42)
all_metrics = []

for fold, (train_idx, val_idx) in enumerate(mskf.split(X, y)):
    print(f"\n===== Fold {fold + 1} =====")
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = EmotionDataset(train_df, tokenizer, label_names)
    val_dataset = EmotionDataset(val_df, tokenizer, label_names)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

    # ---- re-init model each fold ----
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(label_names),
        problem_type="multi_label_classification"
    )
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)
    num_epochs = 3
    num_training_steps = num_epochs * len(train_loader)
    lr_scheduler = get_scheduler("linear", optimizer=optimizer,
                                 num_warmup_steps=0,
                                 num_training_steps=num_training_steps)

    # ---- training loop ----
    model.train()
    progress_bar = tqdm(range(num_training_steps))
    for epoch in range(num_epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

    # ---- evaluation ----
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch["labels"].cpu().numpy().astype(int)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            outputs = model(**inputs)
            logits = outputs.logits.cpu()
            preds = (torch.sigmoid(logits).numpy() > 0.5).astype(int)
            all_preds.append(preds)
            all_labels.append(labels)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    # ---- Global metrics ----
    global_metrics = {
        "subset_acc": accuracy_score(y_true, y_pred),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }
    print("Global metrics:", global_metrics)

    # ---- Per-label metrics ----
    report = classification_report(
        y_true, y_pred,
        target_names=label_names,
        zero_division=0,
        output_dict=True
    )
    print("Classification report:", report)

    # Convert to DataFrame
    df_report = pd.DataFrame(report).transpose()
    df_report["fold"] = fold + 1   # keep track of which fold


    # Save report for later
    all_metrics.append((global_metrics, df_report))

# ===== Average results =====
# Separate global and per-label results
global_results = [m[0] for m in all_metrics]
label_reports  = [m[1] for m in all_metrics]

# Global averages
df_global = pd.DataFrame(global_results)
print("\n===== Global metrics averaged across folds =====")
print(df_global.mean())

# Per-label reports
df_labels = pd.concat



===== Fold 1 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 369/369 [04:48<00:00,  1.65it/s]

Global metrics: {'subset_acc': 0.38738738738738737, 'micro_precision': 0.7323943661971831, 'macro_precision': 0.2445752733389403, 'micro_recall': 0.416, 'macro_recall': 0.156006006006006, 'micro_f1': 0.5306122448979592, 'macro_f1': 0.15781010544614468}
Classification report: {'Joy': {'precision': 0.7560975609756098, 'recall': 0.8378378378378378, 'f1-score': 0.7948717948717948, 'support': 37.0}, 'Trust': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 15.0}, 'Fear': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2.0}, 'Surprise': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 6.0}, 'Sadness': {'precision': 0.6896551724137931, 'recall': 0.6666666666666666, 'f1-score': 0.6779661016949152, 'support': 30.0}, 'Disgust': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 5.0}, 'Anger': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}, 'Anticipation': {'precision': 1.0, 'recall': 0.05555555555555555, 'f1-score': 0.1052631

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


100%|██████████| 369/369 [04:50<00:00,  1.27it/s]


  0%|          | 1/369 [00:01<06:19,  1.03s/it]

  1%|          | 2/369 [00:02<06:21,  1.04s/it]

  1%|          | 3/369 [00:02<05:30,  1.11it/s]

  1%|          | 4/369 [00:03<05:10,  1.18it/s]

  1%|▏         | 5/369 [00:04<04:49,  1.26it/s]

  2%|▏         | 6/369 [00:05<04:40,  1.29it/s]

  2%|▏         | 7/369 [00:05<04:29,  1.35it/s]

  2%|▏         | 8/369 [00:06<04:05,  1.47it/s]

  2%|▏         | 9/369 [00:07<04:22,  1.37it/s]

  3%|▎         | 10/369 [00:07<04:24,  1.36it/s]

  3%|▎         | 11/369 [00:08<04:13,  1.41it/s]

  3%|▎         | 12/369 [00:09<04:33,  1.30it/s]

  4%|▎         | 13/369 [00:10<04:43,  1.26it/s]

  4%|▍        

Global metrics: {'subset_acc': 0.4351851851851852, 'micro_precision': 0.8181818181818182, 'macro_precision': 0.25663865546218484, 'micro_recall': 0.42857142857142855, 'macro_recall': 0.19320933837062867, 'micro_f1': 0.5625, 'macro_f1': 0.21805555555555559}
Classification report: {'Joy': {'precision': 0.7428571428571429, 'recall': 0.7027027027027027, 'f1-score': 0.7222222222222222, 'support': 37.0}, 'Trust': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 15.0}, 'Fear': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2.0}, 'Surprise': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 6.0}, 'Sadness': {'precision': 0.8235294117647058, 'recall': 0.45161290322580644, 'f1-score': 0.5833333333333334, 'support': 31.0}, 'Disgust': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 5.0}, 'Anger': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}, 'Anticipation': {'precision': 1.0, 'recall': 0.7777777777777778, 'f1-score': 0.875

...

Global metrics: {'subset_acc': 0.45217391304347826, 'micro_precision': 0.8571428571428571, 'macro_precision': 0.3655172413793103, 'micro_recall': 0.4838709677419355, 'macro_recall': 0.212012012012012, 'micro_f1': 0.6185567010309279, 'macro_f1': 0.24208574641800826}
Classification report: {'Joy': {'precision': 0.8620689655172413, 'recall': 0.6756756756756757, 'f1-score': 0.7575757575757576, 'support': 37.0}, 'Trust': {'precision': 1.0, 'recall': 0.06666666666666667, 'f1-score': 0.125, 'support': 15.0}, 'Fear': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}, 'Surprise': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 5.0}, 'Sadness': {'precision': 0.7931034482758621, 'recall': 0.7666666666666667, 'f1-score': 0.7796610169491526, 'support': 30.0}, 'Disgust': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 5.0}, 'Anger': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}, 'Anticipation': {'precision': 1.0, 'recall': 0.611111111

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 366/366 [04:43<00:00,  1.29it/s]

100%|██████████| 369/369 [04:30<00:00,  1.38it/s]

Global metrics: {'subset_acc': 0.308411214953271, 'micro_precision': 0.6909090909090909, 'macro_precision': 0.23778735632183906, 'micro_recall': 0.3064516129032258, 'macro_recall': 0.11912912912912912, 'micro_f1': 0.4245810055865922, 'macro_f1': 0.1399222006112809}
Classification report: {'Joy': {'precision': 0.7916666666666666, 'recall': 0.5135135135135135, 'f1-score': 0.6229508196721312, 'support': 37.0}, 'Trust': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 15.0}, 'Fear': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2.0}, 'Surprise': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 5.0}, 'Sadness': {'precision': 0.5862068965517241, 'recall': 0.5666666666666667, 'f1-score': 0.576271186440678, 'support': 30.0}, 'Disgust': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 5.0}, 'Anger': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}, 'Anticipation': {'precision': 1.0, 'recall': 0.1111111111111111, 'f1-score'

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


100%|██████████| 369/369 [04:32<00:00,  1.35it/s]


  0%|          | 1/369 [00:00<05:53,  1.04it/s]

  1%|          | 2/369 [00:02<06:34,  1.07s/it]

  1%|          | 3/369 [00:02<05:12,  1.17it/s]

  1%|          | 4/369 [00:03<04:26,  1.37it/s]

  1%|▏         | 5/369 [00:04<04:28,  1.36it/s]

  2%|▏         | 6/369 [00:04<04:14,  1.43it/s]

  2%|▏         | 7/369 [00:05<04:02,  1.49it/s]

  2%|▏         | 8/369 [00:06<05:12,  1.15it/s]

  2%|▏         | 9/369 [00:07<04:58,  1.21it/s]

  3%|▎         | 10/369 [00:08<04:51,  1.23it/s]

  3%|▎         | 11/369 [00:08<04:39,  1.28it/s]

  3%|▎         | 12/369 [00:09<04:11,  1.42it/s]

  4%|▎         | 13/369 [00:09<03:49,  1.55it/s]

  4%|▍        

Global metrics: {'subset_acc': 0.42990654205607476, 'micro_precision': 0.7714285714285715, 'macro_precision': 0.32272895810159435, 'micro_recall': 0.42857142857142855, 'macro_recall': 0.17788525367472735, 'micro_f1': 0.5510204081632653, 'macro_f1': 0.20256551773251674}
Classification report: {'Joy': {'precision': 0.8709677419354839, 'recall': 0.7297297297297297, 'f1-score': 0.7941176470588235, 'support': 37.0}, 'Trust': {'precision': 1.0, 'recall': 0.06666666666666667, 'f1-score': 0.125, 'support': 15.0}, 'Fear': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2.0}, 'Surprise': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 6.0}, 'Sadness': {'precision': 0.6896551724137931, 'recall': 0.6666666666666666, 'f1-score': 0.6779661016949152, 'support': 30.0}, 'Disgust': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 5.0}, 'Anger': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}, 'Anticipation': {'precision': 0.6666666666666666, 'r

## Cross validation with more information first run

### Global metrics averaged across folds

| Metric           | Value   |
|------------------|---------|
| subset_acc       |  0.414494 |
| micro_precision  |  0.770307 |
| micro_recall     |  0.439963 |
| micro_f1         |  0.558795 |
| macro_f1         |  0.206968 |

### Fold 1
#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.5045  |
| Micro Precision  | 0.7931  |
| Micro Recall     | 0.5520  |
| Micro F1         | 0.6509  |
| Macro F1         | 0.2431  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.8049    | 0.8919 | 0.8462   | 37      |
| Trust         | 1.0000    | 0.0667 | 0.1250   | 15      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 2       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 6       |
| Sadness       | 0.7576    | 0.8333 | 0.7937   | 30      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 3       |
| Anticipation  | 0.8333    | 0.5556 | 0.6667   | 18      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 9       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.7931    | 0.5520 | 0.6509   | 125     |
| Macro avg     | 0.3396    | 0.2347 | 0.2431   | 125     |
| Weighted avg  | 0.6601    | 0.5520 | 0.5519   | 125     |
| Samples avg   | 0.6171    | 0.5676 | 0.5826   | 125     |

---

### Fold 2

#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.4907  |
| Micro Precision  | 0.8333  |
| Micro Recall     | 0.5159  |
| Micro F1         | 0.6373  |
| Macro F1         | 0.2778  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.7222    | 0.7027 | 0.7123   | 37      |
| Trust         | 1.0000    | 0.4000 | 0.5714   | 15      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 2       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 6       |
| Sadness       | 0.8889    | 0.7742 | 0.8276   | 31      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 3       |
| Anticipation  | 1.0000    | 0.5000 | 0.6667   | 18      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 9       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.8333    | 0.5159 | 0.6373   | 126     |
| Macro avg     | 0.3611    | 0.2377 | 0.2778   | 126     |
| Weighted avg  | 0.6927    | 0.5159 | 0.5761   | 126     |
| Samples avg   | 0.5972    | 0.5509 | 0.5648   | 126     |

---

### Fold 3

#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.4167  |
| Micro Precision  | 0.8310  |
| Micro Recall     | 0.4758  |
| Micro F1         | 0.6051  |
| Macro F1         | 0.2055  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.8571    | 0.8108 | 0.8333   | 37      |
| Trust         | 0.0000    | 0.0000 | 0.0000   | 16      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 2       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 5       |
| Sadness       | 0.7742    | 0.8000 | 0.7869   | 30      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 3       |
| Anticipation  | 1.0000    | 0.2778 | 0.4348   | 18      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 8       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.8310    | 0.4758 | 0.6051   | 124     |
| Macro avg     | 0.2631    | 0.1889 | 0.2055   | 124     |
| Weighted avg  | 0.5882    | 0.4758 | 0.5021   | 124     |
| Samples avg   | 0.5278    | 0.4722 | 0.4907   | 124     |

---

### Fold 4

#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.3945  |
| Micro Precision  | 0.7536  |
| Micro Recall     | 0.4160  |
| Micro F1         | 0.5361  |
| Macro F1         | 0.1896  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.8438    | 0.7297 | 0.7826   | 37      |
| Trust         | 0.0000    | 0.0000 | 0.0000   | 15      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 3       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 5       |
| Sadness       | 0.6129    | 0.6129 | 0.6129   | 31      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 2       |
| Anticipation  | 1.0000    | 0.3333 | 0.5000   | 18      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 9       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.7536    | 0.4160 | 0.5361   | 125     |
| Macro avg     | 0.2457    | 0.1676 | 0.1896   | 125     |
| Weighted avg  | 0.5458    | 0.4160 | 0.4557   | 125     |
| Samples avg   | 0.4679    | 0.4312 | 0.4434   | 125     |

---

### Fold 5

#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.4144  |
| Micro Precision  | 0.7333  |
| Micro Recall     | 0.4400  |
| Micro F1         | 0.5500  |
| Macro F1         | 0.2029  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.7273    | 0.6486 | 0.6857   | 37      |
| Trust         | 1.0000    | 0.1250 | 0.2222   | 16      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 3       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 5       |
| Sadness       | 0.6944    | 0.8333 | 0.7576   | 30      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 2       |
| Anticipation  | 1.0000    | 0.2222 | 0.3636   | 18      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 9       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.7333    | 0.4400 | 0.5500   | 125     |
| Macro avg     | 0.3422    | 0.1829 | 0.2029   | 125     |
| Weighted avg  | 0.6539    | 0.4400 | 0.4656   | 125     |
| Samples avg   | 0.4910    | 0.4595 | 0.4685   | 125     |

---

### Fold 6

#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.3945  |
| Micro Precision  | 0.7324  |
| Micro Recall     | 0.4194  |
| Micro F1         | 0.5333  |
| Macro F1         | 0.2005  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.8214    | 0.6216 | 0.7077   | 37      |
| Trust         | 0.0000    | 0.0000 | 0.0000   | 15      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 2       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 5       |
| Sadness       | 0.6207    | 0.6000 | 0.6102   | 30      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 3       |
| Anticipation  | 0.7857    | 0.6111 | 0.6875   | 18      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 9       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.7324    | 0.4194 | 0.5333   | 124     |
| Macro avg     | 0.2228    | 0.1833 | 0.2005   | 124     |
| Weighted avg  | 0.5093    | 0.4194 | 0.4586   | 124     |
| Samples avg   | 0.4771    | 0.4358 | 0.4495   | 124     |

---

### Fold 7

#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.4000  |
| Micro Precision  | 0.7429  |
| Micro Recall     | 0.4094  |
| Micro F1         | 0.5279  |
| Macro F1         | 0.2011  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.7500    | 0.7105 | 0.7297   | 38      |
| Trust         | 1.0000    | 0.0667 | 0.1250   | 15      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 3       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 6       |
| Sadness       | 0.6538    | 0.5484 | 0.5965   | 31      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 2       |
| Anticipation  | 1.0000    | 0.3889 | 0.5600   | 18      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 9       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.7429    | 0.4094 | 0.5279   | 127     |
| Macro avg     | 0.3404    | 0.1714 | 0.2011   | 127     |
| Weighted avg  | 0.6439    | 0.4094 | 0.4581   | 127     |
| Samples avg   | 0.4952    | 0.4460 | 0.4619   | 127     |

---

### Fold 8

#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.3913  |
| Micro Precision  | 0.7500  |
| Micro Recall     | 0.4113  |
| Micro F1         | 0.5313  |
| Macro F1         | 0.1718  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.7692    | 0.8108 | 0.7895   | 37      |
| Trust         | 0.0000    | 0.0000 | 0.0000   | 15      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 3       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 5       |
| Sadness       | 0.6923    | 0.6000 | 0.6429   | 30      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 3       |
| Anticipation  | 1.0000    | 0.1667 | 0.2857   | 18      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 8       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.7500    | 0.4113 | 0.5313   | 124     |
| Macro avg     | 0.2462    | 0.1577 | 0.1718   | 124     |
| Weighted avg  | 0.5422    | 0.4113 | 0.4326   | 124     |
| Samples avg   | 0.4435    | 0.4174 | 0.4261   | 124     |

---

### Fold 9

#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.3738  |
| Micro Precision  | 0.7302  |
| Micro Recall     | 0.3710  |
| Micro F1         | 0.4920  |
| Macro F1         | 0.2241  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.7500    | 0.4865 | 0.5902   | 37      |
| Trust         | 1.0000    | 0.3333 | 0.5000   | 15      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 2       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 5       |
| Sadness       | 0.5769    | 0.5000 | 0.5357   | 30      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 3       |
| Anticipation  | 1.0000    | 0.4444 | 0.6154   | 18      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 9       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.7302    | 0.3710 | 0.4920   | 124     |
| Macro avg     | 0.3327    | 0.1764 | 0.2241   | 124     |
| Weighted avg  | 0.6295    | 0.3710 | 0.4555   | 124     |
| Samples avg   | 0.4206    | 0.3972 | 0.4050   | 124     |

---

### Fold 10

#### 📊 Global Metrics

| Metric           | Value   |
|------------------|---------|
| Subset Accuracy  | 0.3645  |
| Micro Precision  | 0.8033  |
| Micro Recall     | 0.3889  |
| Micro F1         | 0.5241  |
| Macro F1         | 0.1532  |

#### 📋 Classification Report

| Class         | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Joy           | 0.8621    | 0.6757 | 0.7576   | 37      |
| Trust         | 0.0000    | 0.0000 | 0.0000   | 15      |
| Fear          | 0.0000    | 0.0000 | 0.0000   | 2       |
| Surprise      | 0.0000    | 0.0000 | 0.0000   | 6       |
| Sadness       | 0.7500    | 0.8000 | 0.7742   | 30      |
| Disgust       | 0.0000    | 0.0000 | 0.0000   | 5       |
| Anger         | 0.0000    | 0.0000 | 0.0000   | 3       |
| Anticipation  | 0.0000    | 0.0000 | 0.0000   | 19      |
| Neutral       | 0.0000    | 0.0000 | 0.0000   | 9       |
| Reject        | 0.0000    | 0.0000 | 0.0000   | 0       |

#### 📊 Averages

| Metric        | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Micro avg     | 0.8033    | 0.3889 | 0.5241   | 126     |
| Macro avg     | 0.1612    | 0.1476 | 0.1532   | 126     |
| Weighted avg  | 0.4317    | 0.3889 | 0.4068   | 126     |
| Samples avg   | 0.4579    | 0.4097 | 0.4252   | 126     |


In [1]:
import pandas as pd

df = pd.read_csv("../GroundTruth.csv", sep=";")

label_columns = ["Joy", "Trust", "Fear", "Surprise", "Sadness",
                 "Disgust", "Anger", "Anticipation", "Neutral", "Reject"]

non_label_columns = [col for col in df.columns if col not in label_columns]

for label in label_columns:
    df_label = df[non_label_columns + [label]].copy()
    df_label = df_label.rename(columns={label: "Emotion"})
    output_filename = f"{label}.csv"
    df_label.to_csv(output_filename, sep=";", index=False)

    print(f"Archivo generado: {output_filename}")


Archivo generado: Joy.csv
Archivo generado: Trust.csv
Archivo generado: Fear.csv
Archivo generado: Surprise.csv
Archivo generado: Sadness.csv
Archivo generado: Disgust.csv
Archivo generado: Anger.csv
Archivo generado: Anticipation.csv
Archivo generado: Neutral.csv
Archivo generado: Reject.csv


In [2]:
import pandas as pd
from skmultilearn.model_selection import iterative_train_test_split

label_name = "Trust"
df = pd.read_csv(f"{label_name}.csv", sep=";")

label = df[["Emotion"]].values

train_df, train_label, test_df, test_label = iterative_train_test_split(df.values, label, test_size=0.2)

# Convert back to DataFrame and assign columns
train_df = pd.DataFrame(train_df, columns=df.columns)
test_df = pd.DataFrame(test_df, columns=df.columns)

# Assign the label column (flatten in case it's 2D)
train_df["Emotion"] = train_label.ravel()
test_df["Emotion"] = test_label.ravel()

train_df.to_csv("train-joy.csv", index=False)
test_df.to_csv("test-joy.csv", index=False)

# Optional: Check label balance
print("Train label distribution:\n", train_df["Emotion"].sum())
print("Test label distribution:\n", test_df["Emotion"].sum())


Train label distribution:
 122
Test label distribution:
 30


In [7]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

emotions = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral"]

all_results = []

for emotion in emotions:
    print("----------------------------------------------------")
    print(f"Processing label: {emotion}")
    print("----------------------------------------------------")

    # Load data
    df = pd.read_csv('Neutral.csv', sep=';')
    label_col = 'Emotion'
    num_labels = df[label_col].nunique()

    # Tokenizer and collator
    checkpoint = "bert-base-cased"
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Dataset for single-label
    class EmotionDataset(Dataset):
        def __init__(self, df, tokenizer, label_col):
            self.df = df.reset_index(drop=True)
            self.tokenizer = tokenizer
            self.label_col = label_col

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
            tokenized["labels"] = int(row[self.label_col])
            return tokenized

    def collate_fn(batch):
        input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
        labels = [ex["labels"] for ex in batch]
        batch_inputs = data_collator(input_features)
        batch_inputs["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch_inputs

    # Cross-validation
    X = df["sentence"].values
    y = df[label_col].values
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    all_metrics = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n===== Fold {fold + 1} =====")
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)

        train_dataset = EmotionDataset(train_df, tokenizer, label_col)
        val_dataset = EmotionDataset(val_df, tokenizer, label_col)

        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint,
            num_labels=num_labels
        )
        model.to(device)

        optimizer = AdamW(model.parameters(), lr=2e-5)
        num_epochs = 3
        num_training_steps = num_epochs * len(train_loader)
        lr_scheduler = get_scheduler("linear", optimizer=optimizer,
                                     num_warmup_steps=0,
                                     num_training_steps=num_training_steps)

        model.train()
        progress_bar = tqdm(range(num_training_steps))
        for epoch in range(num_epochs):
            for batch in train_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
                progress_bar.update(1)

        # Evaluation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                labels = batch["labels"].cpu().numpy()
                inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
                outputs = model(**inputs)
                preds = outputs.logits.argmax(dim=-1).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(labels)

        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_labels)

        global_metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "micro-precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
            "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "micro-recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
            "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "micro-f1": f1_score(y_true, y_pred, average="micro", zero_division=0)
        }
        print(f"Accuracy  -> {global_metrics['accuracy']}")
        print(f"Precision -> Macro: {global_metrics['precision']}, Micro: {global_metrics['micro-precision']}")
        print(f"Recall    -> Macro: {global_metrics['recall']}, Micro: {global_metrics['micro-recall']}")
        print(f"F1 Score  -> Macro: {global_metrics['f1']}, Micro: {global_metrics['micro-f1']}")

        result_row = {"emotion": emotion, "fold": fold+1}
        result_row.update(global_metrics)
        all_results.append(result_row)

        report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
        df_report = pd.DataFrame(report).transpose()
        df_report["fold"] = fold + 1
        all_metrics.append((global_metrics, df_report))



    # Average results
    global_results = [m[0] for m in all_metrics]
    df_global = pd.DataFrame(global_results)
    print(f"\n===== Global metrics averaged across folds {emotion}=====")
    print(df_global.mean())

    df_summary = pd.DataFrame(all_results)
    print("\n===== All metrics for all emotions and folds =====")
    print(df_summary)

    # If you want the average per emotion:
    df_avg = df_summary.groupby("emotion").mean(numeric_only=True)
    print("\n===== Average metrics per emotion =====")
    print(df_avg)

----------------------------------------------------
Processing label: Joy
----------------------------------------------------


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



===== Fold 1 =====


100%|██████████| 336/336 [03:11<00:00,  2.31it/s]Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Global metrics fold 1:
Accuracy  -> 0.9192825112107623
Precision -> Macro: 0.7154377880184332, Micro: 0.9192825112107623
Recall    -> Macro: 0.5760162601626017, Micro: 0.9192825112107623
F1 Score  -> Macro: 0.6036729857819905, Micro: 0.9192825112107623

===== Fold 2 =====


...

Global metrics fold 3:
Accuracy  -> 0.9234234234234234
Precision -> Macro: 0.7175925925925926, Micro: 0.9234234234234234
Recall    -> Macro: 0.5809182209469154, Micro: 0.9234234234234234
F1 Score  -> Macro: 0.6102447588557265, Micro: 0.9234234234234234

===== Fold 4 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 336/336 [03:17<00:00,  1.79it/s]Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Global metrics fold 4:
Accuracy  -> 0.9054054054054054
Precision -> Macro: 0.5899532710280374, Micro: 0.9054054054054054
Recall    -> Macro: 0.5441893830703013, Micro: 0.9054054054054054
F1 Score  -> Macro: 0.5549403341288782, Micro: 0.9054054054054054

===== Fold 5 =====



100%|██████████| 336/336 [03:21<00:00,  1.67it/s]

100%|██████████| 336/336 [03:18<00:00,  2.19it/s]

Global metrics fold 5:
Accuracy  -> 0.9099099099099099
Precision -> Macro: 0.5860091743119267, Micro: 0.9099099099099099
Recall    -> Macro: 0.5204248366013072, Micro: 0.9099099099099099
F1 Score  -> Macro: 0.5217578629900904, Micro: 0.9099099099099099

===== Global metrics averaged across folds Trust=====
accuracy           0.914564
precision          0.665250
micro-precision    0.914564
recall             0.564093
micro-recall       0.914564
f1                 0.584141
micro-f1           0.914564
dtype: float64

===== All metrics for all emotions and folds =====
  emotion  fold  accuracy  precision  micro-precision    recall  micro-recall  \
0     Joy     1  0.919283   0.715438         0.919283  0.576016      0.919283   
1     Joy     2  0.919283   0.459641         0.919283  0.500000      0.919283   
2     Joy     3  0.932432   0.803241         0.932432  0.612769      0.932432   
3     Joy     4  0.914414   0.669340         0.914414  0.603013      0.914414   
4     Joy     5  0.90991

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split

# Leer dataset
df = pd.read_csv("../GroundTruth.csv", sep=";", quotechar='"')

# Usamos la columna "sentence" como texto y "Emotion" como label
X = df["sentence"]
y = df["Emotion"]

# Hacemos split 80%-20%, estratificado por clase
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Reconstruimos los DataFrames
train_df = pd.DataFrame({"sentence": X_train, "label": y_train})
test_df  = pd.DataFrame({"sentence": X_test, "label": y_test})

# Guardar en CSV
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

# Comprobar distribución de clases
print("Distribución en train:\n", train_df["label"].value_counts())
print("\nDistribución en test:\n", test_df["label"].value_counts())

# Tokenizer and collator
checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        label = row["Emotion"]
        if isinstance(label, str):
            label = self.label_names.index(label)
        tokenized["labels"] = label
        return tokenized

